In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor

In [1]:
import pandas as pd

fc_df = pd.read_csv("fc_HL60.csv")

print(fc_df.shape)
fc_df.head()
fc_df.info()

(46055, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46055 entries, 0 to 46054
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     46055 non-null  object 
 1   GeneName      46055 non-null  object 
 2   Phosphosite   46055 non-null  object 
 3   Perturbation  46055 non-null  object 
 4   CellLine      46055 non-null  object 
 5   FC            46055 non-null  float64
dtypes: float64(1), object(5)
memory usage: 2.1+ MB


In [3]:
ksea_df = pd.read_csv("ksea_HL60.csv")

print(ksea_df.shape)
ksea_df.head()
ksea_df.info()

(13725, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13725 entries, 0 to 13724
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     13725 non-null  object 
 1   GeneName      13725 non-null  object 
 2   Perturbation  13725 non-null  object 
 3   CellLine      13725 non-null  object 
 4   KSEA_z_score  10370 non-null  float64
dtypes: float64(1), object(4)
memory usage: 536.3+ KB


In [6]:
# Download all kinases from API

url = "https://kinepik.org/api/0/kinases/all"

response = requests.get(url)

# Convert API response to DataFrame
kinase_df = pd.DataFrame(response.json())

# Keep only UniProt ID and Gene Symbol
kinase_df = pd.DataFrame({
    "UniprotID": kinase_df["UniprotID"],
    "GeneName": kinase_df["GeneInfo"].apply(lambda x: x["MappedGene"])
})

# Display table
kinase_df.head()

,UniprotID,GeneName
0,P06239,LCK
1,P12931,SRC
2,P06241,FYN
3,P00519,ABL1
4,P24941,CDK2


In [4]:
# Get target phosphosites for one kinase

def get_target_phosphosites(kinase_id):

    url = (
        "https://kinepik.org/api/0/kinases/specific?"
        f"kinase_ids={kinase_id}&phosphosites=targets"
    )

    response = requests.get(url)
    data = response.json()

    if len(data) == 0:
        return []

    return data[0]["TargetPhosphosites"]

In [7]:
# Collect target phosphosites for all kinases

all_target_phosphosites = []

for _, row in kinase_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    targets = get_target_phosphosites(kinase_id)

    all_target_phosphosites.append({

        "UniprotID": kinase_id,
        "GeneName": gene,
        "TargetPhosphositeCount": len(targets),
        "TargetPhosphosites": targets

    })

target_df = pd.DataFrame(all_target_phosphosites)

target_df.head()

,UniprotID,GeneName,TargetPhosphositeCount,TargetPhosphosites
0,P06239,LCK,204,"[SOCS3(Y204), SOCS3(Y221), ESR1(Y537), EZR(Y14..."
1,P12931,SRC,947,"[TERT(Y707), PLSCR1(Y74), PLSCR1(Y69), PDPK1(Y..."
2,P06241,FYN,332,"[FYB1(Y651), FYB1(Y595), FYB1(Y697), FYB1(Y625..."
3,P00519,ABL1,320,"[PLSCR1(Y74), PLSCR1(Y69), TP73(Y99), ANXA1(Y2..."
4,P24941,CDK2,1216,"[TP73(T86), TK1(S13), PGR(S676), PGR(S400), PG..."


In [8]:
target_df.to_csv(
    "target_phosphosite_counts.csv",
    index=False
)

print("Saved target_phosphosite_counts.csv")

Saved target_phosphosite_counts.csv
